# Fastag Fraud Detection

This notebook trains and compares 6 classifiers to detect fraudulent Fastag transactions, with fixes for issues found during review:

1. **Stratified train/test split** — preserves the fraud ratio in both sets.
2. **No leakage in categorical encoding** — encoders are fit on the training split only; unseen categories map to a safe "unknown" code instead of crashing.
3. **Class imbalance is actually handled** — `class_weight='balanced'` where supported, `sample_weight` for Gradient Boosting (which has no `class_weight` parameter).
4. **Real metrics, not hardcoded ones** — precision/recall/F1 on the **Fraud class specifically** (accuracy is misleading here since ~80% of transactions are Not Fraud).
5. **The saved model is the exact one that was evaluated** — no silent "retrain on everything before shipping" step.
6. **Encoders + column order are saved alongside the model**, so the Streamlit app never hand-types category mappings that can drift out of sync.
7. **No leaky derived features** — an earlier version added `Payment_Diff`/`Payment_Ratio`, but this dataset's `Fraud_indicator` turned out to be generated by an exact rule (`Fraud ⟺ Transaction_Amount ≠ Amount_paid`, zero exceptions in all 5,000 rows). Those two engineered columns were removed since they just handed the model the answer key.

Run the cells top to bottom. The last cell saves 4 files needed by `app.py`: `fastag_fraud_model.pkl`, `label_encoders.pkl`, `target_encoder.pkl`, `feature_columns.pkl`.

## 1. Imports

In [1]:
import numpy as np
import pandas as pd
import joblib

from sklearn.base import clone
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import make_scorer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score,
    precision_score, recall_score, f1_score, roc_auc_score,
)

RANDOM_STATE = 42

## 2. Load the data

Put `FastagFraudDetection.csv` in the same folder as this notebook (or edit the path below).

In [2]:
df = pd.read_csv("FastagFraudDetection.csv")
df.head()

,Transaction_ID,Timestamp,Vehicle_Type,FastagID,TollBoothID,Lane_Type,Vehicle_Dimensions,Transaction_Amount,Amount_paid,Geographical_Location,Vehicle_Speed,Vehicle_Plate_Number,Fraud_indicator
0,1,1/6/2023 11:20,Bus,FTG-001-ABC-121,A-101,Express,Large,350,120,"13.059816123454882, 77.77068662374292",65,KA11AB1234,Fraud
1,2,1/7/2023 14:55,Car,FTG-002-XYZ-451,B-102,Regular,Small,120,100,"13.059816123454882, 77.77068662374292",78,KA66CD5678,Fraud
2,3,1/8/2023 18:25,Motorcycle,NaN,D-104,Regular,Small,0,0,"13.059816123454882, 77.77068662374292",53,KA88EF9012,Not Fraud
3,4,1/9/2023 2:05,Truck,FTG-044-LMN-322,C-103,Regular,Large,350,120,"13.059816123454882, 77.77068662374292",92,KA11GH3456,Fraud
4,5,1/10/2023 6:35,Van,FTG-505-DEF-652,B-102,Express,Medium,140,100,"13.059816123454882, 77.77068662374292",60,KA44IJ6789,Fraud


## 3. Feature engineering

- `state_code` — first two characters of the vehicle plate number (proxy for registration state).
- `Hour`, `Day_of_Week`, `Month` — extracted from the timestamp.
- Drop identifier / free-text columns (`Transaction_ID`, `FastagID`, `Geographical_Location`, `Vehicle_Plate_Number`, `Timestamp`) that don't generalize.

**Note:** `Payment_Diff`/`Payment_Ratio` are deliberately *not* created here — see the intro cell for why.

In [3]:
df["state_code"] = df["Vehicle_Plate_Number"].str[:2]
df.drop(columns=["Vehicle_Plate_Number"], inplace=True)

df["Timestamp"] = pd.to_datetime(df["Timestamp"])
df["Hour"] = df["Timestamp"].dt.hour
df["Day_of_Week"] = df["Timestamp"].dt.dayofweek
df["Month"] = df["Timestamp"].dt.month
df.drop(columns=["Timestamp"], inplace=True)

df.drop(columns=["Transaction_ID", "FastagID", "Geographical_Location"], inplace=True)

df.head()

,Vehicle_Type,TollBoothID,Lane_Type,Vehicle_Dimensions,Transaction_Amount,Amount_paid,Vehicle_Speed,Fraud_indicator,state_code,Hour,Day_of_Week,Month
0,Bus,A-101,Express,Large,350,120,65,Fraud,KA,11,4,1
1,Car,B-102,Regular,Small,120,100,78,Fraud,KA,14,5,1
2,Motorcycle,D-104,Regular,Small,0,0,53,Not Fraud,KA,18,6,1
3,Truck,C-103,Regular,Large,350,120,92,Fraud,KA,2,0,1
4,Van,B-102,Express,Medium,140,100,60,Fraud,KA,6,1,1


## 4. Encode the target and split the data

In [4]:
categorical_cols = ["Vehicle_Type", "Lane_Type", "Vehicle_Dimensions",
                     "TollBoothID", "state_code"]
target_col = "Fraud_indicator"

X = df.drop(columns=[target_col])
y_raw = df[target_col]

# Target encoding: fit on the full label set (only 2 fixed classes -- no
# leakage concern the way per-row features have).
target_encoder = LabelEncoder()
y = target_encoder.fit_transform(y_raw)
print("Target classes:", list(target_encoder.classes_),
      "-> codes", list(range(len(target_encoder.classes_))))

Target classes: ['Fraud', 'Not Fraud'] -> codes [0, 1]


In [5]:
# FIX: stratified split -- keeps the fraud ratio the same in train and test
x_train, x_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print("Train size:", x_train.shape, " Test size:", x_test.shape)

Train size: (4000, 11)  Test size: (1000, 11)


## 5. Encode categorical columns (leakage-safe)

Each `LabelEncoder` is fit **only** on `x_train`. Any category seen in `x_test` (or later, at inference time in the app) that wasn't in the training data maps to a reserved "unknown" code instead of raising a `KeyError`.

In [6]:
label_encoders = {}
for col in categorical_cols:
    x_train[col] = x_train[col].astype(str)
    x_test[col] = x_test[col].astype(str)

    le = LabelEncoder()
    le.fit(x_train[col])
    class_to_code = {cls: code for code, cls in enumerate(le.classes_)}
    unknown_code = len(class_to_code)  # reserved for categories not seen in training

    x_train[col] = x_train[col].map(class_to_code)
    x_test[col] = x_test[col].map(class_to_code).fillna(unknown_code).astype(int)

    label_encoders[col] = {"class_to_code": class_to_code, "unknown_code": unknown_code}

feature_columns = list(x_train.columns)  # exact order app.py must replicate
print("Feature columns:", feature_columns)

Feature columns: ['Vehicle_Type', 'TollBoothID', 'Lane_Type', 'Vehicle_Dimensions', 'Transaction_Amount', 'Amount_paid', 'Vehicle_Speed', 'state_code', 'Hour', 'Day_of_Week', 'Month']


## 6. Define models and train (imbalance-aware)

- `class_weight='balanced'` for models that support it.
- `sample_weight` (via `compute_sample_weight`) for Gradient Boosting, which has no `class_weight` parameter.
- KNN has no native class-weighting option, so it's left as-is.

In [7]:
sample_weight_train = compute_sample_weight(class_weight="balanced", y=y_train)
FRAUD_CODE = int(np.where(target_encoder.classes_ == "Fraud")[0][0])

candidates = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "Decision Tree": DecisionTreeClassifier(class_weight="balanced", random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(class_weight="balanced", random_state=RANDOM_STATE),
    "Gradient Boosting": GradientBoostingClassifier(random_state=RANDOM_STATE),  # no class_weight arg
    "SVM": SVC(class_weight="balanced", probability=True, random_state=RANDOM_STATE),
    "KNN": KNeighborsClassifier(),  # no native class-weighting option
}

## 7. Evaluation helper

Reports precision/recall/F1 **for the Fraud class specifically** (not just overall accuracy, which is misleading on an ~80/20 imbalanced dataset), plus a correctly-oriented ROC-AUC.

> Fix note: `roc_auc_score` treats the numerically larger label (`1` = "Not Fraud" in our encoding) as positive by default. Since we care about ranking the *Fraud* class, we relabel explicitly before scoring.

In [8]:
def evaluate_model(name, y_true, y_pred, y_score=None):
    result = {
        "model": name,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision_fraud": precision_score(y_true, y_pred, pos_label=FRAUD_CODE),
        "recall_fraud": recall_score(y_true, y_pred, pos_label=FRAUD_CODE),
        "f1_fraud": f1_score(y_true, y_pred, pos_label=FRAUD_CODE),
    }
    if y_score is not None:
        y_true_fraud = (np.asarray(y_true) == FRAUD_CODE).astype(int)
        result["roc_auc_fraud"] = roc_auc_score(y_true_fraud, y_score)
    print(f"\n--- {name} ---")
    print(classification_report(y_true, y_pred, target_names=list(target_encoder.classes_)))
    print("Confusion matrix:\n", confusion_matrix(y_true, y_pred))
    return result

## 8. Fit each model and evaluate on the held-out test set

In [9]:
results, fitted_models = [], {}
for name, model in candidates.items():
    if name == "Gradient Boosting":
        model.fit(x_train, y_train, sample_weight=sample_weight_train)
    else:
        model.fit(x_train, y_train)

    y_pred = model.predict(x_test)
    y_score = model.predict_proba(x_test)[:, FRAUD_CODE] if hasattr(model, "predict_proba") else None
    results.append(evaluate_model(name, y_test, y_pred, y_score))
    fitted_models[name] = model

results_df = pd.DataFrame(results)


--- Logistic Regression ---
              precision    recall  f1-score   support

       Fraud       1.00      0.94      0.97       197
   Not Fraud       0.99      1.00      0.99       803

    accuracy                           0.99      1000
   macro avg       0.99      0.97      0.98      1000
weighted avg       0.99      0.99      0.99      1000

Confusion matrix:
 [[185  12]
 [  0 803]]

--- Decision Tree ---
              precision    recall  f1-score   support

       Fraud       1.00      0.99      0.99       197
   Not Fraud       1.00      1.00      1.00       803

    accuracy                           1.00      1000
   macro avg       1.00      0.99      1.00      1000
weighted avg       1.00      1.00      1.00      1000

Confusion matrix:
 [[195   2]
 [  0 803]]

--- Random Forest ---
              precision    recall  f1-score   support

       Fraud       1.00      0.93      0.96       197
   Not Fraud       0.98      1.00      0.99       803

    accuracy           

/home/ravi-kumar-pal/ML Internship/Machine Learning Internship MIP-ML-07/ML internship/Fastag_Fraud_detection/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(



--- SVM ---
              precision    recall  f1-score   support

       Fraud       1.00      0.90      0.95       197
   Not Fraud       0.98      1.00      0.99       803

    accuracy                           0.98      1000
   macro avg       0.99      0.95      0.97      1000
weighted avg       0.98      0.98      0.98      1000

Confusion matrix:
 [[177  20]
 [  0 803]]

--- KNN ---
              precision    recall  f1-score   support

       Fraud       1.00      0.95      0.97       197
   Not Fraud       0.99      1.00      0.99       803

    accuracy                           0.99      1000
   macro avg       0.99      0.97      0.98      1000
weighted avg       0.99      0.99      0.99      1000

Confusion matrix:
 [[187  10]
 [  0 803]]


## 9. Cross-validate to avoid trusting one lucky split

If several models land at or near a perfect fraud-F1 on a single test split, that's more likely a red flag (leaky feature, or a small/easy test set) than a sign they're all great. Running 5-fold cross-validation on the training data checks whether performance holds up consistently across folds.

In [10]:
fraud_f1_scorer = make_scorer(f1_score, pos_label=FRAUD_CODE)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

cv_rows = []
for name, model in candidates.items():
    cv_model = clone(model)  # fresh, unfit copy -- CV must not reuse fitted state
    scores = cross_val_score(cv_model, x_train, y_train, cv=cv, scoring=fraud_f1_scorer)
    cv_rows.append({"model": name, "cv_f1_fraud_mean": scores.mean(), "cv_f1_fraud_std": scores.std()})
    print(f"{name:22s} 5-fold CV fraud-F1: {scores.mean():.3f} (+/- {scores.std():.3f})  folds={np.round(scores, 3)}")

cv_df = pd.DataFrame(cv_rows)
results_df = results_df.merge(cv_df, on="model")
results_df = results_df.sort_values(["cv_f1_fraud_mean", "f1_fraud"], ascending=False)

print("\n=== Model comparison (sorted by cross-validated fraud-F1, then test-set fraud-F1) ===")
results_df

Logistic Regression    5-fold CV fraud-F1: 0.980 (+/- 0.010)  folds=[0.97  0.99  0.967 0.987 0.987]
Decision Tree          5-fold CV fraud-F1: 0.992 (+/- 0.008)  folds=[1.    0.99  0.977 0.997 0.997]
Random Forest          5-fold CV fraud-F1: 0.975 (+/- 0.009)  folds=[0.974 0.984 0.96  0.984 0.974]
Gradient Boosting      5-fold CV fraud-F1: 0.979 (+/- 0.014)  folds=[0.984 0.984 0.953 0.994 0.981]


/home/ravi-kumar-pal/ML Internship/Machine Learning Internship MIP-ML-07/ML internship/Fastag_Fraud_detection/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/ravi-kumar-pal/ML Internship/Machine Learning Internship MIP-ML-07/ML internship/Fastag_Fraud_detection/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/ravi-kumar-pal/ML Internship/Machine Learning Internship MIP-ML-07/ML internship/Fastag_Fraud_detection/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will b

SVM                    5-fold CV fraud-F1: 0.965 (+/- 0.012)  folds=[0.957 0.974 0.946 0.977 0.971]
KNN                    5-fold CV fraud-F1: 0.975 (+/- 0.007)  folds=[0.97  0.97  0.967 0.981 0.984]

=== Model comparison (sorted by cross-validated fraud-F1, then test-set fraud-F1) ===


,model,accuracy,precision_fraud,recall_fraud,f1_fraud,roc_auc_fraud,cv_f1_fraud_mean,cv_f1_fraud_std
1,Decision Tree,0.998,1.0,0.989848,0.994898,0.994924,0.992237,0.008147
0,Logistic Regression,0.988,1.0,0.939086,0.968586,0.966604,0.980445,0.009642
3,Gradient Boosting,0.991,1.0,0.954315,0.976623,0.998954,0.979041,0.013570
2,Random Forest,0.986,1.0,0.928934,0.963158,0.998413,0.975157,0.008658
5,KNN,0.990,1.0,0.949239,0.973958,0.984743,0.974506,0.006509
4,SVM,0.980,1.0,0.898477,0.946524,0.964467,0.964972,0.011627


In [11]:
near_perfect = results_df[results_df["cv_f1_fraud_mean"] >= 0.999]
if len(near_perfect) > 0:
    print(
        f"\n\u26a0\ufe0f  {len(near_perfect)} model(s) scored ~1.00 fraud-F1 across CV folds: "
        f"{list(near_perfect['model'])}. Real-world fraud detection is essentially "
        "never perfectly separable -- check the feature-importance cell below. "
        "If Transaction_Amount/Amount_paid dominate, remember this dataset defines "
        "Fraud_indicator as an exact rule on those two columns, so a model finding "
        "it isn't learning a subtle pattern -- it's finding the label formula."
    )
else:
    print("No model scored suspiciously perfect -- results look realistic.")

No model scored suspiciously perfect -- results look realistic.


## 10. Select and save the best model

Important: the model saved below is the **exact object that was just evaluated** (trained only on `x_train`) — not a version silently retrained on the full dataset afterward. That means the metrics above genuinely describe what gets deployed.

In [12]:
best_name = results_df.iloc[0]["model"]
best_model = fitted_models[best_name]
print(f"Selected model: {best_name} "
      f"(CV fraud-F1={results_df.iloc[0]['cv_f1_fraud_mean']:.3f}, "
      f"test fraud-F1={results_df.iloc[0]['f1_fraud']:.3f}, "
      f"test recall-fraud={results_df.iloc[0]['recall_fraud']:.3f})")

Selected model: Decision Tree (CV fraud-F1=0.992, test fraud-F1=0.995, test recall-fraud=0.990)


## 11. Explain the model: feature importance

Check whether `Transaction_Amount`/`Amount_paid` dominate — that would be consistent with the known leakage rule in this dataset (`Fraud ⟺ Transaction_Amount ≠ Amount_paid`), rather than a sign the model found a subtle, generalizable pattern.

In [13]:
if hasattr(best_model, "feature_importances_"):
    importances = pd.Series(best_model.feature_importances_, index=feature_columns)
    importances = importances.sort_values(ascending=False)
    print("Feature importances (top 5):")
    print(importances.head(5).to_string())
elif hasattr(best_model, "coef_"):
    coefs = pd.Series(best_model.coef_[0], index=feature_columns).sort_values(key=abs, ascending=False)
    print("Model coefficients (top 5 by magnitude):")
    print(coefs.head(5).to_string())
else:
    print(f"{best_name} doesn't expose feature_importances_ or coef_.")

Feature importances (top 5):
Amount_paid           0.464663
Transaction_Amount    0.455138
TollBoothID           0.070274
Hour                  0.003501
Vehicle_Speed         0.001954


## 12. Save the model + encoders for the Streamlit app

`app.py` loads these four files directly — no hand-typed category mappings, no guessing column order.

In [14]:
joblib.dump(best_model, "fastag_fraud_model.pkl")
joblib.dump(label_encoders, "label_encoders.pkl")
joblib.dump(target_encoder, "target_encoder.pkl")
joblib.dump(feature_columns, "feature_columns.pkl")

print("Saved: fastag_fraud_model.pkl, label_encoders.pkl, target_encoder.pkl, feature_columns.pkl")
print("\nNext step: run the Streamlit app with:  streamlit run app.py")

Saved: fastag_fraud_model.pkl, label_encoders.pkl, target_encoder.pkl, feature_columns.pkl

Next step: run the Streamlit app with:  streamlit run app.py
